# R3 com limiar fixo comum

Recalcula a regra R3 com um único limiar de densidade (mediana global: 139,86 termos por 10 mil tokens), comum a todas as categorias. As regras R1 e R2 são recalculadas como referência e reproduzem os valores da Tabela 5.2 da dissertação.

**Dados de entrada:** corpus pré-processado em `legislatura/data/artigo1_preprocessado`, gerado por `scripts/01_preprocess_corpus.py` a partir dos dados da API da Câmara dos Deputados (ver `data/README_DATA.md`). O notebook procura essa pasta no Google Drive montado.

**Execução:** Google Colab → *Ambiente de execução → Executar tudo*.

**Dicionários:** este notebook utiliza a versão completa dos dicionários. Os resultados com a versão final, adotada na dissertação, são gerados pelo notebook `23_camadas1_2_dicionario_final.ipynb`.

In [ ]:
# 1) Conectar o Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Localizar a pasta pré-processada da dissertação
from pathlib import Path
RAIZ = Path('/content/drive/MyDrive')
candidatos = [RAIZ / 'legislatura' / 'data' / 'artigo1_preprocessado']
candidatos += list(RAIZ.glob('*/legislatura/data/artigo1_preprocessado'))
candidatos += list(RAIZ.glob('*/data/artigo1_preprocessado'))
PRE = next((c for c in candidatos if (c / 'corpus_artigo1_metadata.csv').exists()), None)
if PRE is None:
    raise SystemExit('Pasta artigo1_preprocessado não encontrada no Drive montado (ver Dados de entrada no topo).')
print('Pasta encontrada:', PRE)

SAIDA = RAIZ / 'MESTRADO' / 'resultados_R3_dissertacao'
SAIDA.mkdir(parents=True, exist_ok=True)
PERCENTIL_R3 = 0.50   # percentil do limiar fixo da R3 (0.50 = mediana)

In [ ]:
# 3) Ler os discursos pré-processados (mesma leitura do script 14)
import pandas as pd
from collections import Counter
LEGS = [52, 53, 54, 55, 56, 57]
meta = pd.read_csv(PRE / 'corpus_artigo1_metadata.csv', usecols=['id_legislatura'])
textos, legs = [], []
for leg in LEGS:
    linhas = (PRE / 'by_legislature' / f'leg{leg}.txt').read_text(encoding='utf-8').splitlines()
    n_meta = int((meta['id_legislatura'] == leg).sum())
    if n_meta != len(linhas):
        raise ValueError(f'Leg {leg}: metadados={n_meta} texto={len(linhas)}')
    textos += linhas; legs += [leg] * len(linhas)
    print(f'Legislatura {leg}: {len(linhas):,} discursos')
df = pd.DataFrame({'_leg': legs, '_tokens': [t.split() for t in textos]})
del textos, meta
print(f'TOTAL: {len(df):,} discursos  (dissertação: 439.627)')

In [ ]:
# 4) Dicionários temáticos (idênticos aos dos scripts 08 e 14)
THEMES = {
 "Educação": {"educacao","escola","escolas","ensino","professor","professores","aluno","alunos","universidade","universidades","creche","alfabetizacao","fundeb","enem","estudante","estudantes","pesquisa","ciencia","cientifico","tecnologia"},
 "Saúde": {"saude","sus","hospital","hospitais","medico","medicos","enfermeiro","enfermeiros","enfermeiras","vacina","vacinas","pandemia","covid","doenca","doencas","tratamento","medicamento","medicamentos","pacientes","atendimento"},
 "Economia, trabalho e renda": {"economia","economico","economica","emprego","empregos","trabalho","trabalhador","trabalhadores","renda","salario","salarios","inflacao","juros","credito","industria","empresas","empresa","mercado","crescimento","desenvolvimento","investimento","investimentos","tributaria","imposto","impostos"},
 "Segurança pública e violência": {"seguranca","violencia","crime","crimes","criminalidade","policia","policial","policiais","prisao","presidio","armas","arma","trafico","homicidio","homicidios","feminicidio","militar","militares"},
 "Meio ambiente e clima": {"meio","ambiente","ambiental","amazonia","floresta","florestas","desmatamento","clima","climatica","climatico","sustentavel","sustentabilidade","biodiversidade","indigena","indigenas","garimpo","queimadas","agua","saneamento"},
 "Infraestrutura, transporte e habitação": {"infraestrutura","transporte","transportes","rodovia","rodovias","estrada","estradas","ferrovia","ferrovias","porto","portos","aeroporto","aeroportos","energia","eletrica","habitacao","moradia","saneamento","internet","telecomunicacoes","obra","obras"},
 "Proteção social e desigualdade": {"pobreza","fome","desigualdade","desigualdades","assistencia","social","bolsa","familia","beneficio","beneficios","previdencia","aposentadoria","aposentados","vulnerabilidade","moradia","direitos","mulheres","criancas","juventude","idosos","deficiencia"},
 "Federalismo e municípios": {"municipio","municipios","prefeito","prefeitos","prefeitura","prefeituras","estado","estados","federacao","federal","governador","governadores","regional","regionais","regiao","regioes","nordeste","norte","sul","sudeste","centro"},
 "Democracia e instituições": {"democracia","democratico","democratica","constituicao","constitucional","congresso","camara","senado","parlamento","governo","presidente","presidencia","republica","eleicao","eleicoes","voto","impeachment","stf","supremo","judiciario","justica"},
 "Direitos e cidadania": {"direito","direitos","cidadania","cidadao","cidadaos","igualdade","liberdade","justica","mulher","mulheres","negros","racismo","discriminacao","lgbt","indigena","indigenas","quilombolas","criancas","adolescentes"},
 "Agricultura e alimentação": {"agricultura","agricola","agropecuaria","agronegocio","produtor","produtores","rural","campo","alimentos","alimento","comida","fome","pecuaria","safra","assentamento","reforma","agraria"},
}
print(len(THEMES), 'temas')

In [ ]:
# 5) Contagem por discurso e tema
import numpy as np
rows = []
for i, toks in enumerate(df["_tokens"]):
    c = Counter(toks); n = len(toks)
    for tema, termos in THEMES.items():
        hits = [t for t in termos if c[t] > 0]
        total = sum(c[t] for t in hits)
        rows.append((i, tema, len(hits), total, 10_000 * total / n if n else 0.0))
    if i % 50_000 == 0:
        print(f"  {i:,}/{len(df):,}")
s = pd.DataFrame(rows, columns=["discurso","tema","termos_distintos","ocorrencias","densidade_10k"])
del rows
N = len(df)
print("Pronto.")

In [ ]:
# 6) Regras R1, R2, R3 com limiar por categoria e R3 com limiar fixo
s["R1"] = s["ocorrencias"] > 0
s["R2"] = (s["termos_distintos"] >= 2) | (s["ocorrencias"] >= 3)

# R3 com limiar por categoria: mediana POR TEMA entre os discursos R1 (reproduz a Tabela 5.2 da dissertação)
med_tema = s[s["R1"]].groupby("tema")["densidade_10k"].median()
s["R3_limiar_por_categoria"] = s["R1"] & (s["densidade_10k"] >= s["tema"].map(med_tema))

# R3 com limiar fixo: limiar FIXO, igual para todos os temas
LIMIAR = s.loc[s["R1"], "densidade_10k"].quantile(PERCENTIL_R3)
s["R3_limiar_fixo"] = s["R1"] & (s["densidade_10k"] >= LIMIAR)
print(f"Limiar fixo da R3 com limiar fixo: {LIMIAR:.2f} ocorrências por 10 mil tokens (percentil {PERCENTIL_R3:.2f})")

tab = s.groupby("tema")[["R1","R2","R3_limiar_por_categoria","R3_limiar_fixo"]].sum()
pct = (100 * tab / N).round(1)
pct["R3_limiar_por_categoria/R1"] = (tab["R3_limiar_por_categoria"] / tab["R1"]).round(3)
pct["R3_limiar_fixo/R1"]   = (tab["R3_limiar_fixo"] / tab["R1"]).round(3)
pct = pct.sort_values("R1", ascending=False)
pct

In [ ]:
# 7) A ordenação dos temas se mantém?
from scipy.stats import spearmanr
ordem = pd.DataFrame({r: pct[r].rank(ascending=False) for r in ["R1","R2","R3_limiar_fixo"]})
print("Correlação de Spearman entre as ordenações:")
print("  R1 x R2     :", round(spearmanr(pct["R1"], pct["R2"]).correlation, 3))
print("  R1 x R3 com limiar fixo:", round(spearmanr(pct["R1"], pct["R3_limiar_fixo"]).correlation, 3))
print("  R2 x R3 com limiar fixo:", round(spearmanr(pct["R2"], pct["R3_limiar_fixo"]).correlation, 3))
ordem

In [ ]:
# 8) Salvar no Drive (pasta MESTRADO/resultados_R3_dissertacao)
out = pct.reset_index().rename(columns={"index": "tema"})
out.insert(1, "limiar_R3_por_10k", round(LIMIAR, 4))
out.to_csv(SAIDA / "tabela_5_2_R3_limiar_fixo.csv", index=False, encoding="utf-8-sig")

por_leg = s.merge(df[["_leg"]], left_on="discurso", right_index=True)
leg = (100 * por_leg.groupby(["_leg","tema"])[["R1","R2","R3_limiar_fixo"]].sum()
       / por_leg.groupby(["_leg","tema"]).size().values[:, None]).round(1)
leg.to_csv(SAIDA / "R1_R2_R3_por_legislatura.csv", encoding="utf-8-sig")
print("Arquivos salvos em:", SAIDA)